# 03 — Unsupervised Clustering & Anomaly Detection

KMeans, DBSCAN, Hierarchical Clustering → wallet persona.
Isolation Forest, One-Class SVM → outlier/anomaly wallet.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
sns.set_theme(style='whitegrid')

In [ ]:
# Synthetic wallet features
np.random.seed(42)
n = 500
X = np.vstack([
    np.random.randn(n//4, 5) + [0, 5, -2, 1, 10],   # cluster A
    np.random.randn(n//4, 5) + [8, 0, 3, -1, -5],    # cluster B
    np.random.randn(n//4, 5) + [-5, -3, 0, 8, 2],    # cluster C
    np.random.randn(n//4, 5) + [2, 2, -5, -3, 0],    # cluster D
])
X_scaled = StandardScaler().fit_transform(X)
print(f'Data: {X_scaled.shape}')

## KMeans Clustering

In [ ]:
inertias, sil_scores = [], []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(2, 10), inertias, 'o-'); ax1.set_title('Elbow'); ax1.set_xlabel('k')
ax2.plot(range(2, 10), sil_scores, 'o-'); ax2.set_title('Silhouette Score'); ax2.set_xlabel('k')
plt.show()

In [ ]:
best_k = np.argmax(sil_scores) + 2
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_scaled)
coords = PCA(n_components=2).fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=kmeans.labels_, cmap='tab10', alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c='red', marker='x', s=100)
plt.title(f'KMeans (k={best_k}) — PCA projection'); plt.legend(*scatter.legend_elements(), title='Cluster')
plt.show()

## DBSCAN — Density-based

In [ ]:
db = DBSCAN(eps=2.5, min_samples=5).fit(X_scaled)
n_clusters_db = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise = sum(db.labels_ == -1)
print(f'DBSCAN: {n_clusters_db} clusters, {n_noise} noise points')

## Anomaly Detection — Isolation Forest + One-Class SVM

In [ ]:
iforest = IsolationForest(contamination=0.05, random_state=42).fit(X_scaled)
ocsvm = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale').fit(X_scaled)
if_anom = iforest.predict(X_scaled) == -1
ocsvm_anom = ocsvm.predict(X_scaled) == -1
print(f'Isolation Forest anomalies: {if_anom.sum()}')
print(f'One-Class SVM anomalies: {ocsvm_anom.sum()}')
print(f'Both agree: {(if_anom & ocsvm_anom).sum()}')